In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# 1. Leer los datos
edges_file = 'edges.csv'
features_file = 'features.csv'
edges = pd.read_csv(edges_file)
features = pd.read_csv(features_file)

# 2. Crear los conjuntos de datos
edges_dataset = tf.data.TextLineDataset(edges_file).map(parse_edge)
features_dataset = tf.data.TextLineDataset(features_file).map(parse_features)

# 3. Preprocesar los datos
features_dataset = features_dataset.map(normalize_features)

# 4. Definir el modelo
num_nodes = features.shape[0]
num_classes = 2
hidden_units = 16

# Capa de entrada
inputs = layers.Input(shape=(None,))
# Capa de grafos convolucionales
graph_conv = GraphConvolution(hidden_units)(inputs)
graph_conv = layers.BatchNormalization()(graph_conv)
graph_conv = layers.Activation('relu')(graph_conv)
# Capa de clasificación
outputs = layers.Dense(num_classes, activation='softmax')(graph_conv)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 5. Entrenar el modelo
labels = tf.one_hot(labels, num_classes)
dataset = tf.data.Dataset.zip((features_dataset, edges_dataset)).batch(batch_size)

model.fit(dataset, labels, epochs=num_epochs, validation_split=0.2)

In [31]:
!pip install pandas

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [32]:
import os
import pandas as pd
import numpy as np
#import networkx as nx
#import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [33]:
hidden_units = [32, 32]
learning_rate = 0.01
dropout_rate = 0.5
num_epochs = 2
batch_size = 8 #256


In [34]:
def compile_model(model):
    # Compile the model.
    model.compile(
        optimizer="rmsprop",
        #optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        # Tati: categorical_crossentropy, expects the labels to follow a categorical encoding. 
        #       With integer labels, you should use sparse_categorical_crossentropy.
        #       This new loss function is still mathematically the same as categorical_crossentropy; it just has a different interface.
        #metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    # Create an early stopping callback.
    #early_stopping = keras.callbacks.EarlyStopping(
    #    monitor="val_acc", patience=50, restore_best_weights=True
    #
    #)
    return model

# This function trains an input model using the given training data.
def run_experiment(model, x_train, y_train): 
    guardarModelo = keras.callbacks.ModelCheckpoint(
        filepath="/mnt/modelos1/",  # "checkpoint_path.keras",
        monitor="val_loss",
        save_best_only=True,
        save_format="tf",
    )
    # Fit the model.
    history = model.fit(
        x=x_train,
        y=y_train,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_split=0.15,
        callbacks=[guardarModelo], #early_stopping],
    )

    return history


def create_ffn(hidden_units, dropout_rate, name=None):
    fnn_layers = []

    for units in hidden_units:
        fnn_layers.append(layers.BatchNormalization())
        fnn_layers.append(layers.Dropout(dropout_rate))
        fnn_layers.append(layers.Dense(units, activation=tf.nn.gelu))

    return keras.Sequential(fnn_layers, name=name)


In [35]:
class GraphConvLayer(layers.Layer):
    def __init__(
        self,
        hidden_units,
        dropout_rate=0.2,
        aggregation_type="mean",
        combination_type="concat",
        normalize=False,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.aggregation_type = aggregation_type
        self.combination_type = combination_type
        self.normalize = normalize

        self.ffn_prepare = create_ffn(hidden_units, dropout_rate)
        if self.combination_type == "gated":
            self.update_fn = layers.GRU(
                units=hidden_units,
                activation="tanh",
                recurrent_activation="sigmoid",
                dropout=dropout_rate,
                return_state=True,
                recurrent_dropout=dropout_rate,
            )
        else:
            self.update_fn = create_ffn(hidden_units, dropout_rate)

    def prepare(self, node_repesentations, weights=None):
        # node_repesentations shape is [num_edges, embedding_dim].
        messages = self.ffn_prepare(node_repesentations)
        if weights is not None:
            messages = messages * tf.expand_dims(weights, -1)
        return messages

    def aggregate(self, node_indices, neighbour_messages, node_repesentations):
        # node_indices shape is [num_edges].
        # neighbour_messages shape: [num_edges, representation_dim].
        # node_repesentations shape is [num_nodes, representation_dim]
        num_nodes = node_repesentations.shape[0]
        if self.aggregation_type == "sum":
            aggregated_message = tf.math.unsorted_segment_sum(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "mean":
            aggregated_message = tf.math.unsorted_segment_mean(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "max":
            aggregated_message = tf.math.unsorted_segment_max(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        else:
            raise ValueError(f"Invalid aggregation type: {self.aggregation_type}.")

        return aggregated_message

    def update(self, node_repesentations, aggregated_messages):
        # node_repesentations shape is [num_nodes, representation_dim].
        # aggregated_messages shape is [num_nodes, representation_dim].
        if self.combination_type == "gru":
            # Create a sequence of two elements for the GRU layer.
            h = tf.stack([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "concat":
            # Concatenate the node_repesentations and aggregated_messages.
            h = tf.concat([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "add":
            # Add node_repesentations and aggregated_messages.
            h = node_repesentations + aggregated_messages
        else:
            raise ValueError(f"Invalid combination type: {self.combination_type}.")

        # Apply the processing function.
        node_embeddings = self.update_fn(h)
        if self.combination_type == "gru":
            node_embeddings = tf.unstack(node_embeddings, axis=1)[-1]

        if self.normalize:
            node_embeddings = tf.nn.l2_normalize(node_embeddings, axis=-1)
        return node_embeddings

    def call(self, inputs):
        """Process the inputs to produce the node_embeddings.

        inputs: a tuple of three elements: node_repesentations, edges, edge_weights.
        Returns: node_embeddings of shape [num_nodes, representation_dim].
        """

        node_repesentations, edges, edge_weights = inputs
        # Get node_indices (source) and neighbour_indices (target) from edges.
        node_indices, neighbour_indices = edges[0], edges[1]
        # neighbour_repesentations shape is [num_edges, representation_dim].
        neighbour_repesentations = tf.gather(node_repesentations, neighbour_indices)

        # Prepare the messages of the neighbours.
        neighbour_messages = self.prepare(neighbour_repesentations, edge_weights)
        # Aggregate the neighbour messages.
        aggregated_messages = self.aggregate(
            node_indices, neighbour_messages, node_repesentations
        )
        # Update the node embedding with the neighbour messages.
        return self.update(node_repesentations, aggregated_messages)


In [36]:
class GNNNodeClassifier(tf.keras.Model):
    def __init__(
        self,
        graph_info,
        num_classes,
        hidden_units,
        aggregation_type="sum",
        combination_type="concat",
        dropout_rate=0.2,
        normalize=True,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        # Unpack graph_info to three elements: node_features, edges, and edge_weight.
        node_features, edges, edge_weights = graph_info
        self.node_features = node_features
        self.edges = edges
        self.edge_weights = edge_weights
        # Set edge_weights to ones if not provided.
        if self.edge_weights is None:
            self.edge_weights = tf.ones(shape=edges.shape[1])
        # Scale edge_weights to sum to 1.
        self.edge_weights = self.edge_weights / tf.math.reduce_sum(self.edge_weights)

        # Create a process layer.
        self.preprocess = create_ffn(hidden_units, dropout_rate, name="preprocess")
        # Create the first GraphConv layer.
        self.conv1 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv1",
        )
        # Create the second GraphConv layer.
        self.conv2 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv2",
        )
        # Create a postprocess layer.
        self.postprocess = create_ffn(hidden_units, dropout_rate, name="postprocess")
        # Create a compute logits layer.
        self.compute_logits = layers.Dense(units=num_classes, name="logits")  
          # Tati: For Tensorflow: logits is a name that it is thought to imply that this Tensor is the quantity that is being mapped to probabilities by the Softmax

    def call(self, input_node_indices):
        # Preprocess the node_features to produce node representations.
        x = self.preprocess(self.node_features)
        # Apply the first graph conv layer.
        x1 = self.conv1((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x1 + x
        # Apply the second graph conv layer.
        x2 = self.conv2((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x2 + x
        # Postprocess node embedding.
        x = self.postprocess(x)
        # Fetch node embeddings for the input node_indices.
        node_embeddings = tf.gather(x, input_node_indices)
        # Compute logits
        return self.compute_logits(node_embeddings)


In [19]:
## ESTO NO
def node_dict(file):
    # Lee archivo
    usecols = ["node","ID","OD","IDW","ODW","label"]
    features = pd.read_csv(file, sep=",", header=0, usecols=lambda c: c in set(usecols))
    node_idx = {name: idx for idx, name in enumerate(sorted(features["node"].unique()))}
    class_idx = {name: idx for idx, name in enumerate(sorted(features["label"].unique()))}
    return (node_idx, class_idx)

In [14]:
## ESTO NO
def parse_features(file, node_idx):
    # Lee archivo
    usecols = ["node","ID","OD","IDW","ODW","label"]
    features = pd.read_csv(file, sep=",", header=0, usecols=lambda c: c in set(usecols))
    
    # Genera diccionario para las clases y nodos
    class_idx = {name: idx for idx, name in enumerate(sorted(features["label"].unique()))}
    #node_idx = {name: idx for idx, name in enumerate(sorted(features["node"].unique()))}
    
    #features = features_tmp.loc[:,["node","ID","OD","IDW","ODW","label"]].copy()
    features["node"] = features["node"].apply(lambda name: node_idx[name])
    features["label"] = features["label"].apply(lambda value: class_idx[value])
    feature_names = set(features.columns) - {"node", "label"}
    node_features = tf.cast(features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32) ## CAMBIAR POR float16 ??
    node_labels = tf.cast(features["label"], dtype=tf.dtypes.int64)
    #dict_final = {'node_features': node_features, 'node_labels': node_labels}
    #return dict_final
    final = (node_features, node_labels)
    return final


In [7]:
## ESTO NO
def parse_features(line):
    feature_description = {
        "node": tf.io.FixedLenFeature([], tf.int64),
        "ID": tf.io.FixedLenFeature([], tf.int64),
        "OD": tf.io.FixedLenFeature([], tf.int64),
        "IDW": tf.io.FixedLenFeature([], tf.int64),
        "ODW": tf.io.FixedLenFeature([], tf.int64),
        "label": tf.io.FixedLenFeature([], tf.int64)
    }
    example = tf.io.parse_single_example(line, feature_description)    
    
    return {
        "node_int": example["node"],
        "node_ID": example["ID"],
        "node_OD": example["OD"],
        "node_IDW": example["IDW"],
        "node_ODW": example["ODW"],
        "node_label": example["label"]
    }

def parse_edge(line):
    feature_description = {
        "source": tf.io.FixedLenFeature([], tf.int64),
        "target": tf.io.FixedLenFeature([], tf.int64),
        "weight": tf.io.FixedLenFeature([], tf.float32),
    }
    example = tf.io.parse_single_example(line, feature_description)    
    
    return {
        "node_source": example["source"],
        "node_target": example["target"],
        "edge_weight": example["weight"]
    }


In [ ]:
## ESTO NO
training_grafos = pd.read_csv(
    "training_GRAFOS.pkts.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features_tmp = pd.read_csv(
    "training_FEATURES.pkts.SINnorm.csv",
    sep=",",  
    header=0
)


In [9]:
## ESTO NO
def parse_edge(file, node_idx):
    grafos = pd.read_csv(file, sep=" ", header=None, names=["source", "target", "weight"],)
    grafos["source"] = grafos["source"].apply(lambda name: node_idx[name])
    grafos["target"] = grafos["target"].apply(lambda name: node_idx[name])
    edges = grafos[["source", "target"]].to_numpy().T
    edges_weights = grafos[["weight"]].to_numpy().T 
    edges_weights = edges_weights.reshape((edges.shape[-1],))
    edges_weights = tf.convert_to_tensor(edges_weights)
    #dict_final = {'edges': edges, 'edges_weights': edges_weights}
    #return dict_final
    final = (edges, edges_weights)
    return final


In [10]:
## ESTO NO
edges_file = "/mnt/training_GRAFOS_INT.pkts.ncol"
features_file = "/mnt/training_FEATURES_INT.pkts.SINnorm.csv"

#node_idx = node_dict(features_file)
edges_dataset = tf.data.TextLineDataset(edges_file).map(parse_edge)
features_dataset = tf.data.TextLineDataset(features_file).map(parse_features)

#dataset = tf.data.Dataset.from_tensor_slices((images, labels))



In [15]:
features_types = [int(), int(), int(), int(), int(), int()]

simple_features = tf.data.experimental.CsvDataset(features_file, record_defaults=features_types, header=True)

for element in simple_features.take(5):
    print([e.numpy() for e in element])



[0, 0, 1, 0, 2239, 1]
[252661, 6, 0, 2907, 0, 1]
[1, 1, 1, 6, 6, 1]
[97219, 525631, 540374, 4796296, 4936679, 1]
[2, 1, 1, 2, 2, 1]


In [18]:
edges_types = [int(), int(), float()]

simple_edge = tf.data.experimental.CsvDataset(edges_file, record_defaults=edges_types, field_delim=" ")

for element in simple_edge.take(5):
    print([e.numpy() for e in element])


[0, 252661, 2239.0]
[1, 97219, 6.0]
[2, 97219, 2.0]
[3, 97219, 1.0]
[4, 97219, 1.0]


In [ ]:
## ESTO NO
edges_file = "/mnt/training_GRAFOS.pkts.ncol"
features_file = "/mnt/training_FEATURES.pkts.SINnorm.csv"
#edges = pd.read_csv(edges_file)
#features = pd.read_csv(features_file)

training_grafos = pd.read_csv(
    edges_file,
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features_tmp = pd.read_csv(
    features_file,
    sep=",",  
    header=0
)

edges_dataset = tf.data.TextLineDataset(edges_file).map(parse_edge)
features_dataset = tf.data.TextLineDataset(features_file).map(parse_features)

labels = tf.one_hot(labels, num_classes)
dataset = tf.data.Dataset.zip((features_dataset, edges_dataset)).batch(batch_size)

model.fit(dataset, labels, epochs=num_epochs, validation_split=0.2)

In [37]:
## ESTO SI

edges_file = "/mnt/training_GRAFOS_INT.pkts.ncol"
features_file = "/mnt/training_FEATURES_INT.pkts.SINnorm.csv"

features_types = [int(), int(), int(), int(), int(), int()]
features_dataset = tf.data.experimental.CsvDataset(features_file, record_defaults=features_types, header=True)
for element in features_dataset.take(5):
    print([e.numpy() for e in element])
    
print("="*10)

edges_types = [int(), int(), float()]
edges_dataset = tf.data.experimental.CsvDataset(edges_file, record_defaults=edges_types, field_delim=" ")
for element in edges_dataset.take(5):
    print([e.numpy() for e in element])

[0, 0, 1, 0, 2239, 1]
[252661, 6, 0, 2907, 0, 1]
[1, 1, 1, 6, 6, 1]
[97219, 525631, 540374, 4796296, 4936679, 1]
[2, 1, 1, 2, 2, 1]
[0, 252661, 2239.0]
[1, 97219, 6.0]
[2, 97219, 2.0]
[3, 97219, 1.0]
[4, 97219, 1.0]


In [38]:
dataset = tf.data.Dataset.zip((features_dataset, edges_dataset))

In [46]:
for element in dataset.take(1):
    print(element)
    #print([e.numpy() for e in element])
    #for e in element.take(1):
    #    print(e.numpy())

((<tf.Tensor: shape=(), dtype=int32, numpy=0>, <tf.Tensor: shape=(), dtype=int32, numpy=0>, <tf.Tensor: shape=(), dtype=int32, numpy=1>, <tf.Tensor: shape=(), dtype=int32, numpy=0>, <tf.Tensor: shape=(), dtype=int32, numpy=2239>, <tf.Tensor: shape=(), dtype=int32, numpy=1>), (<tf.Tensor: shape=(), dtype=int32, numpy=0>, <tf.Tensor: shape=(), dtype=int32, numpy=252661>, <tf.Tensor: shape=(), dtype=float32, numpy=2239.0>))


In [30]:
dataset.take(1)

TypeError: 'TakeDataset' object is not subscriptable